# 04 · Analysis: ablation (RQ2), diagnostics, and generalization (RQ4)

* **RQ2** – feature-family *ablation* (drop one) and *standalone* (only one), as in Hard to Park Tables 2–3.
  Uses logistic regression because it is fast and interpretable; repeat with the MLP if time allows.
* **Diagnostics** – confusion matrices, performance by slot and by ticker.
* **RQ4** – train on tech, test on financials (and optionally crypto), against a locally trained model and the baseline.

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from htt import config as C

import json
from htt import modeling as M, evaluation as E
ds = pd.read_parquet(C.DATA_PROC / "dataset.parquet")
fam = json.load(open(C.DATA_PROC / "feature_families.json"))

STOCKS = ds[ds.sector.isin(["tech", "fin"])]
tr, va, te = [STOCKS[STOCKS.split == s] for s in ("train", "val", "test")]

def fit_eval(cols, tr=tr, va=va, te=te):
    m = M.RewardDecisionLR().fit(tr[cols].values, tr.y.values, va[cols].values, va.y.values)
    return E.bnr(te.y.values, m.predict(te[cols].values))

ref_bnr = E.bnr(te.y.values, M.HistoricalBaseline().fit(tr, tr.y).predict(te))
full_bnr = fit_eval(fam["all"])
print(f"baseline BNR {ref_bnr:.3f} | full LR BNR {full_bnr:.3f}")

## RQ2 · Feature-family ablation

In [ ]:
families = ["spread_liquidity", "price_deviation", "oscillation", "busyness", "calendar"]
rows = []
for f in families:
    drop = [c for c in fam["all"] if c not in fam[f]]
    rows.append(dict(family=f, only_this=fit_eval(fam[f]) - full_bnr, without_this=fit_eval(drop) - full_bnr))
abl = pd.DataFrame(rows).set_index("family")
abl.round(3)   # relative BNR vs the full model; closer to 0 in "without_this" = less important

In [ ]:
abl.plot(kind="bar", figsize=(8, 3.5), title="Relative BNR vs full model (LR)"); plt.axhline(0, c="k", lw=.5); plt.tight_layout()
abl.to_csv(C.RESULTS / "ablation.csv")

Note the paper's caveat: families can carry **redundant** information (distance, vicinity and deviation times),
so removing one rarely hurts much while using one alone can still be strong. Expect the same here:
last week's effective spread (in `spread_liquidity`) is likely dominant, so also try an ablation of
**single raw features** within that family.

## Diagnostics

In [ ]:
pred = pd.read_parquet(C.RESULTS / "test_predictions.parquet")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, k in zip(ax, ["baseline_ticker_slot", "logreg_L2", "mlp_reward"]):
    E.plot_confusion(pred.y, pred[k], k, ax=a)
plt.tight_layout()

In [ ]:
by_slot = pd.DataFrame({k: pred.groupby("slot").apply(lambda g: E.bnr(g.y, g[k]))
                        for k in ["baseline_ticker_slot", "logreg_L2", "mlp_reward"]})
by_slot.plot(marker="o", figsize=(7, 3.5), title="Test BNR by slot"); plt.tight_layout()
by_tick = pred.groupby("symbol").apply(lambda g: E.bnr(g.y, g.mlp_reward) - E.bnr(g.y, g.baseline_ticker_slot)).sort_values()
by_tick.plot(kind="bar", figsize=(10, 3), title="MLP minus baseline BNR, by ticker (test)"); plt.tight_layout()

### Interpretability: largest logistic-regression coefficients for the *hard* class

In [ ]:
lr = M.RewardDecisionLR().fit(tr[fam["all"]].values, tr.y.values, va[fam["all"]].values, va.y.values)
coef = pd.Series(lr.model.coef_[2], index=fam["all"]).sort_values()
pd.concat([coef.head(8), coef.tail(8)]).plot(kind="barh", figsize=(7, 5), title="LR coefficients (hard class)"); plt.tight_layout()

## RQ4 · Generalization

Because labels and features are both **per-ticker relative**, a model trained on one group can score another.
We compare, on the *financials test split*:

1. baseline (own history), 2. model trained on **tech** only, 3. model trained on **financials** (local).

The calendar one-hots are removed for the crypto experiment because crypto has 24 slots/day, 7 days/week
(no open/close) and slot indices do not mean the same thing.

In [ ]:
def split_of(sector):
    d = ds[ds.sector == sector]
    return [d[d.split == s] for s in ("train", "val", "test")]

def transfer(src, dst, cols, label):
    s_tr, s_va, _ = split_of(src); d_tr, d_va, d_te = split_of(dst)
    mk = lambda tr_, va_: M.RewardDecisionLR().fit(tr_[cols].values, tr_.y.values, va_[cols].values, va_.y.values)
    xfer, local = mk(s_tr, s_va), mk(d_tr, d_va)
    base = M.HistoricalBaseline().fit(d_tr, d_tr.y)
    y = d_te.y.values
    return dict(setting=label,
                baseline=E.bnr(y, base.predict(d_te)),
                transferred=E.bnr(y, xfer.predict(d_te[cols].values)),
                local=E.bnr(y, local.predict(d_te[cols].values)))

no_cal = [c for c in fam["all"] if c not in fam["calendar"]]
res = [transfer("tech", "fin", fam["all"], "tech -> fin (all features)"),
       transfer("tech", "fin", no_cal, "tech -> fin (no calendar)")]
if (ds.sector == "crypto").any():
    res.append(transfer("tech", "crypto", no_cal, "tech -> crypto (no calendar)"))
    res.append(transfer("fin", "crypto", no_cal, "fin -> crypto (no calendar)"))
gen = pd.DataFrame(res).set_index("setting")
gen["transfer_gap"] = gen.local - gen.transferred
gen.round(3)

In [ ]:
gen[["baseline", "transferred", "local"]].plot(kind="bar", figsize=(9, 3.8), title="Test BNR on target domain"); plt.tight_layout()
gen.to_csv(C.RESULTS / "generalization.csv")

**Reading it:** `transferred ≥ baseline` means a model learned on one market segment still adds value elsewhere;
`transfer_gap` is how much local training would buy, directly comparable to the 0.06–0.10 BNR gap the
Hard to Park paper reports between transferred and locally trained models.